# Project 3 — Movie Recommender: Build & Evaluate

**Duration:** 5 min

## Load Data & Build Model

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv('ml-100k/u.data', sep='\t', names=['userId','movieId','rating','ts'])
movies = pd.read_csv('ml-100k/u.item', sep='|', encoding='latin-1',
                     usecols=[0,1], names=['movieId','title'])

# Re-index users and movies to 0-based
df['userId']  = df['userId']  - 1
df['movieId'] = df['movieId'] - 1
n_users, n_movies = df['userId'].nunique(), df['movieId'].nunique()

class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.tensor(df['userId'].values,  dtype=torch.long)
        self.movies = torch.tensor(df['movieId'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
    def __len__(self): return len(self.ratings)
    def __getitem__(self, i): return self.users[i], self.movies[i], self.ratings[i]

class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, n_factors=50):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users,  n_factors)
        self.movie_emb = nn.Embedding(n_movies, n_factors)
        self.user_bias  = nn.Embedding(n_users,  1)
        self.movie_bias = nn.Embedding(n_movies, 1)
    def forward(self, u, m):
        dot = (self.user_emb(u) * self.movie_emb(m)).sum(1)
        return dot + self.user_bias(u).squeeze() + self.movie_bias(m).squeeze()

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-projects-intermediate/ip-project3-build.ipynb)


## Train & Recommend

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MatrixFactorization(n_users, n_movies).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-5)
criterion = nn.MSELoss()

for epoch in range(20):
    model.train()
    for u, m, r in train_loader:
        u, m, r = u.to(device), m.to(device), r.to(device)
        optimizer.zero_grad()
        criterion(model(u, m), r).backward()
        optimizer.step()

# Evaluate
model.eval()
with torch.no_grad():
    u = torch.tensor(test_df['userId'].values).to(device)
    m = torch.tensor(test_df['movieId'].values).to(device)
    preds = model(u, m).cpu().numpy()
rmse = np.sqrt(((preds - test_df['rating'].values)**2).mean())
print(f'Test RMSE: {rmse:.4f}')

def recommend(user_id, n=10):
    model.eval()
    seen = set(df[df['userId']==user_id]['movieId'])
    all_movies = torch.arange(n_movies).to(device)
    u = torch.tensor([user_id]*n_movies).to(device)
    with torch.no_grad():
        scores = model(u, all_movies).cpu().numpy()
    scores[list(seen)] = -999
    top = np.argsort(scores)[::-1][:n]
    return movies[movies['movieId'].isin(top+1)]['title'].tolist()

print(recommend(0))

> **💡 Tip:** ✅ Project 3 complete! You've built a production-style recommender system. Claim your certificate!